# 02 · Semantic Search via Public Ollama API

Run this notebook in **Colab B** (or on another computer). It calls the public Pinggy URL of Notebook 1. No local embedding model is installed here.

**Tasks:** Connect → Create embeddings → Cosine similarity → Top-k search → Try new queries.

### 1. Paste your Pinggy HTTPS URL

In [1]:
import requests
import numpy as np

PUBLIC_URL = "https://PASTE_YOUR_PINGGY_LINK_HERE"
MODEL = "embeddinggemma"
PUBLIC_URL = PUBLIC_URL.rstrip("/")

response = requests.get(f"{PUBLIC_URL}/api/tags", timeout=30)
response.raise_for_status()
print("Connected! Ollama models:", [m["name"] for m in response.json().get("models", [])])

Connected! Ollama models: ['embeddinggemma:latest']


### 2. Call the embedding API

In [2]:
def embed(texts):
    response = requests.post(
        f"{PUBLIC_URL}/api/embed",
        json={"model": MODEL, "input": texts},
        timeout=120,
    )
    response.raise_for_status()
    return np.array(response.json()["embeddings"], dtype=float)

vector = embed(["How can I get a new student card?"])[0]
print("Dimension:", len(vector))
print("First 8 values:", np.round(vector[:8], 4))

Dimension: 768
First 8 values: [-0.1092 -0.0086  0.0381  0.0071  0.0547  0.0553  0.005  -0.034 ]


### 3. Embed documents (indexing)

In [3]:
documents = [
    "Students who lose their ID card can request a replacement at the Student Services Office.",
    "Course registration is available on the university portal before each semester.",
    "Tuition fees must be paid before the announced deadline.",
    "The library provides access to books, journals, and study rooms.",
    "Graduation requires completing all mandatory courses and earning enough credits.",
    "Exam schedules are published on the student portal.",
]

doc_vectors = embed(documents)   # One API request for all documents
print("Embedding matrix shape:", doc_vectors.shape)

Embedding matrix shape: (6, 768)


### 4. Cosine similarity + Top-k retrieval

In [4]:
query = "I lost my campus card. Where can I get another one?"
query_vector = embed([query])[0]

# cosine(q, d) = (q · d) / (||q|| ||d||)
scores = (doc_vectors @ query_vector) / (
    np.linalg.norm(doc_vectors, axis=1) * np.linalg.norm(query_vector) + 1e-12
)

top_k = 3
for i in np.argsort(scores)[::-1][:top_k]:
    print(f"Score: {scores[i]:.4f} | {documents[i]}")

Score: 0.7346 | Students who lose their ID card can request a replacement at the Student Services Office.
Score: 0.3290 | Graduation requires completing all mandatory courses and earning enough credits.
Score: 0.3243 | Exam schedules are published on the student portal.


### 5. Reusable search function

In [5]:
def search(question, k=3):
    q = embed([question])[0]
    sims = (doc_vectors @ q) / (
        np.linalg.norm(doc_vectors, axis=1) * np.linalg.norm(q) + 1e-12
    )
    for i in np.argsort(sims)[::-1][:k]:
        print(f"{sims[i]:.4f} | {documents[i]}")

search("How do I pay for my classes?", k=2)

0.5701 | Tuition fees must be paid before the announced deadline.
0.4458 | Course registration is available on the university portal before each semester.


### 6. Student tasks

1. Change the query and observe the top-3 results.
2. Add two new documents, rerun the document embedding cell, and search again.
3. Compare a query that uses the **same keywords** with one using **paraphrases**.
4. Explain why the document embeddings need not be regenerated for every new query.

In [6]:
# TODO: Try two different queries
search("What do I need in order to graduate?", k=3)
search("Where can I borrow books?", k=3)

0.7172 | Graduation requires completing all mandatory courses and earning enough credits.
0.4691 | Tuition fees must be paid before the announced deadline.
0.4259 | Exam schedules are published on the student portal.
0.5382 | The library provides access to books, journals, and study rooms.
0.4185 | Students who lose their ID card can request a replacement at the Student Services Office.
0.3665 | Exam schedules are published on the student portal.
